# Leaflet cluster map of talk locations

Assuming you are working in a Linux or Windows Subsystem for Linux environment, you may need to install some dependencies. Assuming a clean installation, the following will be needed:

```bash
sudo apt install jupyter
sudo apt install python3-pip
pip install python-frontmatter getorg --upgrade
```

After which you can run this from the `_talks/` directory, via:

```bash
 jupyter nbconvert --to notebook --execute talkmap.ipynb --output talkmap_out.ipynb
```
 
The `_talks/` directory contains `.md` files of all your talks. This scrapes the location YAML field from each `.md` file, geolocates it with `geopy/Nominatim`, and uses the `getorg` library to output data, HTML, and Javascript for a standalone cluster map.

In [1]:
# Start by installing the dependencies
import frontmatter
import os
import glob
import getorg
from geopy import Nominatim
from geopy.exc import GeocoderTimedOut

Iywidgets and ipyleaflet support disabled. You must be in a Jupyter notebook to use this feature.
Error raised:
No module named 'ipywidgets'
Check that you have enabled ipyleaflet in Jupyter with:
    jupyter nbextension enable --py ipyleaflet


In [12]:
# Collect the Markdown files
g = glob.glob("_talks/*.md")
g1 = glob.glob("_teaching/*.md")

g += g1
print(g)

['_talks/2025-09-01-talk-2.md', '_talks/2025-09-26-talk-3.md', '_talks/2025-12-04-talk-4.md', '_talks/2025-09-02-talk-1.md', '_teaching/2024-autumn-teaching-1.md', '_teaching/2025-notte-della-ricerca.md']


In [13]:
# Set the default timeout, in seconds
TIMEOUT = 5

# Prepare to geolocate
geocoder = Nominatim(user_agent="academicpages.github.io")
location_dict = {}
location = ""
permalink = ""
title = ""

In the event that this times out with an error, double check to make sure that the location is can be properly geolocated.

In [14]:
g

['_talks/2025-09-01-talk-2.md',
 '_talks/2025-09-26-talk-3.md',
 '_talks/2025-12-04-talk-4.md',
 '_talks/2025-09-02-talk-1.md',
 '_teaching/2024-autumn-teaching-1.md',
 '_teaching/2025-notte-della-ricerca.md']

In [20]:
# Perform geolocation
for file in g:
    # Read the file
    data = frontmatter.Frontmatter.read_file(file)
    data = data['attributes']

    # Press on if the location is not present
    if 'location' not in data:
        continue

    # Prepare the description
    title = data['title'].strip()
    venue = data['venue'].strip()
    location = data['location'].strip()
    if ',' in location:     # consider only before the second comma in the location, if there is one
        location = ','.join(location.split(',')[0:1]).strip()

    description = f"{title}<br />{venue}; {location}"

    #break
    # Geocode the location and report the status
    try:
        location_dict[description] = geocoder.geocode(location, timeout=TIMEOUT)
        print(location_dict[description])
    except ValueError as ex:
        print(f"Error: geocode failed on input {location} with message {ex}")
    except GeocoderTimedOut as ex:
        print(f"Error: geocode timed out on input {location} with message {ex}")
    except Exception as ex:
        print(f"An unhandled exception occurred while processing input {location} with message {ex}")

Siena, Toscana, Italia
Trento, Territorio Val d'Adige, Provincia di Trento, Trentino-Alto Adige/Südtirol, Italia
Verona, Veneto, Italia
Siena, Toscana, Italia
Trento, Territorio Val d'Adige, Provincia di Trento, Trentino-Alto Adige/Südtirol, Italia
Trento, Territorio Val d'Adige, Provincia di Trento, Trentino-Alto Adige/Südtirol, Italia


In [19]:
data

{'attributes': {'title': 'Come la scienza smaschera la disinformazione',
  'collection': 'teaching',
  'type': 'Evento di divulgazione',
  'permalink': '/teaching/2025-notte-della-ricerca',
  'venue': 'Notte della ricerca, MUSE, Trento, TN',
  'date': datetime.date(2025, 9, 26),
  'location': 'Trento, TN, ITA'},
 'body': '> Siamo circondati da una marea di informazioni potenzialmente vere o false, ma quanto siamo consapevoli della loro natura? Scopriremo insieme quanto i social media siano disseminati di contenuti potenzialmente disinformanti e dannosi e di come rendere più consapevole la propria esperienza digitale. Vieni a scoprire il potere della conoscenza e del pensiero critico per navigare al meglio nel mondo dell\'informazione!\n\n\nChe cosa è la disinformazione?\n------\nLa disinformazione è l\'insieme di informazioni false o fuorvianti diffuse intenzionalmente per manipolare l\'opinione pubblica o influenzare il comportamento delle persone. Può propagarsi attraverso vari canal

In [21]:
# Save the map
m = getorg.orgmap.create_map_obj()

# Clean the output file
if os.path.exists('talkmap/org-locations.js'):
    os.remove('talkmap/org-locations.js')

getorg.orgmap.output_html_cluster_map(location_dict, folder_name="talkmap", hashed_usernames=False)

'Written map to talkmap/'